# Módulo 5 · Modelos de Secuencias: Transformers, LLMs y Modelos de Espacio de Estados (Mamba)

## Objetivos
1. Implementar la **atención escalada por producto punto** y la **atención multi-cabeza** desde cero y entender qué calcula cada matriz $Q, K, V$.
2. Construir un **Transformer decodificador (mini-GPT)** completo: *embeddings*, codificación posicional (sinusoidal y **RoPE**), bloques residuales pre-LN, máscara causal; entrenarlo a nivel de carácter.
3. Entender la **tokenización BPE**, el **KV-cache** y el muestreo (temperatura, top-p) que hacen práctico un LLM.
4. Distinguir las familias: *encoder* (BERT → representaciones), *decoder* (GPT/Llama → generación), *encoder-decoder* (T5), y el ciclo **pre-entrenamiento → SFT → RLHF/DPO**.
5. Implementar un **modelo de espacio de estados selectivo** (estilo Mamba) y comparar su costo $O(L)$ con el $O(L^2)$ de la atención; conocer las arquitecturas **híbridas** que dominan en 2026.
6. Usar **Hugging Face Transformers** para inferencia y *fine-tuning* ligero.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Atención, Transformer, BERT, pre-entrenamiento | Zhang et al., *Dive into Deep Learning* | **Cap. 11** Attention Mechanisms and Transformers · **Cap. 15** NLP: Pretraining · **Cap. 16** NLP: Applications |
| Panorama de Transformers y ecosistema Hugging Face | Tunstall, von Werra & Wolf, *NLP with Transformers* | **Cap. 1** Hello Transformers · **Cap. 2** Text Classification · **Cap. 3** Transformer Anatomy · **Cap. 8** Making Transformers Efficient in Production · **Cap. 10** Training Transformers from Scratch |
| LLMs: tokens, *embeddings*, interior de un LLM | Alammar & Grootendorst, *Hands-On Large Language Models* | **Cap. 1** An Introduction to LLMs · **Cap. 2** Tokens and Embeddings · **Cap. 3** Looking Inside Transformer LLMs |
| Atención y Transformers en Keras, ViT | Géron, *Hands-On ML* (3.ª ed.) | **Cap. 16** NLP with RNNs and Attention (secciones de atención, "An Avalanche of Transformer Models", Vision Transformers) |
| Fundamentos de los modelos de lenguaje grandes | Fregly et al., *Generative AI on AWS* | **Cap. 3** Large-Language Foundation Models |

> **Nota de actualización (2026).** La malla original citaba "Alammar Cap. 2 (Introducción a LLMs)": en el libro el Cap. 1 es la introducción y el **Cap. 2 son Tokens y Embeddings**. LangChain se mueve al Módulo 6 (ingeniería de aplicaciones), y se añaden **RoPE, KV-cache, Mamba/SSM e híbridos**, que no existen en los libros (2022-2024) y son estándar hoy. Mamba-3 (2026) se cita como lectura frontera.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 5 · Transformers, LLMs y Modelos de Espacio de Estados**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, time
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    TORCH = True; torch.manual_seed(0); device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch", torch.__version__, device)
except ImportError:
    TORCH = False; print("PyTorch no instalado; este módulo lo requiere.")

## 1. Atención: una consulta ponderada sobre una memoria (D2L 11.1-11.5; Tunstall Cap. 3)

Dadas consultas $Q\in\mathbb{R}^{n\times d_k}$, claves $K\in\mathbb{R}^{m\times d_k}$ y valores $V\in\mathbb{R}^{m\times d_v}$:
$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V .$$
* Cada fila de $QK^\top$ dice *cuánto se parece* la consulta $i$ a cada clave $j$; el softmax lo convierte en pesos que suman 1; la salida es una **media ponderada de los valores**.
* $\sqrt{d_k}$ evita que los logits crezcan con la dimensión y saturen el softmax.
* **Auto-atención**: $Q, K, V$ son proyecciones lineales de la *misma* secuencia → cada token "mira" a todos los demás en **un solo paso paralelo** (vs. $T$ pasos secuenciales de una RNN).
* **Máscara causal**: en un decodificador, el token $t$ solo puede ver $\le t$ (ponemos $-\infty$ antes del softmax).
* **Multi-cabeza**: $h$ atenciones en subespacios de dimensión $d/h$, concatenadas y proyectadas: distintas cabezas capturan distintas relaciones (sintaxis, correferencia, posición…).

Costo: $O(n^2 d)$ en tiempo y $O(n^2)$ en memoria por la matriz de atención — la razón de FlashAttention (misma matemática, sin materializar la matriz) y de las alternativas lineales (sección 6).

In [ ]:
if TORCH:
    def atencion(Q, K, V, mask=None):
        scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
        if mask is not None: scores = scores.masked_fill(mask, float("-inf"))
        w = scores.softmax(-1)
        return w @ V, w

    # Ejemplo interpretable: 5 tokens, la consulta del token 4 'busca' claves parecidas a la suya
    torch.manual_seed(3); d = 8; X = torch.randn(5, d)
    Wq, Wk, Wv = [torch.randn(d, d) / math.sqrt(d) for _ in range(3)]
    causal = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
    out, w = atencion(X @ Wq, X @ Wk, X @ Wv, mask=causal)
    print("pesos de atención (causal; cada fila suma 1):\n", w.numpy().round(3))
    # Comprobación contra la implementación fusionada de PyTorch (usa FlashAttention/SDPA cuando está disponible)
    ref = F.scaled_dot_product_attention(X @ Wq, X @ Wk, X @ Wv, is_causal=True)
    print("coincide con F.scaled_dot_product_attention:", torch.allclose(out, ref, atol=1e-5))

## 2. Codificación posicional: sinusoidal vs RoPE

La atención es **invariante a permutaciones**: sin información de posición, "perro muerde hombre" = "hombre muerde perro". Dos soluciones:

* **Sinusoidal (Vaswani 2017)**: se *suma* al embedding un vector fijo $PE_{(pos,2i)}=\sin(pos/10000^{2i/d})$, $PE_{(pos,2i+1)}=\cos(\cdot)$.
* **RoPE — Rotary Position Embedding (Su et al., 2021)**: se *rotan* $q$ y $k$ por un ángulo proporcional a la posición, en pares de dimensiones. Resultado: $\langle q_m, k_n\rangle$ depende solo de la **posición relativa** $m-n$. Es el estándar en Llama, Mistral, Qwen, Gemma, DeepSeek… y permite extender el contexto (NTK-aware scaling, YaRN).

Implementamos ambas; verificamos que con RoPE el producto punto solo depende de la distancia relativa.

In [ ]:
if TORCH:
    def pe_sinusoidal(T, d):
        pos = torch.arange(T)[:, None]; i = torch.arange(0, d, 2)[None]
        ang = pos / (10000 ** (i / d)); pe = torch.zeros(T, d); pe[:, 0::2] = ang.sin(); pe[:, 1::2] = ang.cos(); return pe

    def rope(x, pos):
        '''x: (..., T, d) con d par; pos: (T,). Rota pares (x_{2i}, x_{2i+1}) por ángulo pos*theta_i.'''
        d = x.shape[-1]; theta = 10000 ** (-torch.arange(0, d, 2, device=x.device) / d)
        ang = pos[:, None] * theta[None]; cos, sin = ang.cos(), ang.sin()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

    plt.figure(figsize=(9, 3)); plt.imshow(pe_sinusoidal(60, 64).T, aspect="auto", cmap="RdBu"); plt.xlabel("posición"); plt.ylabel("dimensión")
    plt.title("Codificación sinusoidal: frecuencias altas arriba, bajas abajo (un 'reloj' multi-escala)"); plt.colorbar(); plt.show()

    q = torch.randn(1, 16); k = torch.randn(1, 16)
    s1 = (rope(q, torch.tensor([5.])) * rope(k, torch.tensor([2.]))).sum()      # posiciones 5 y 2 (distancia 3)
    s2 = (rope(q, torch.tensor([105.])) * rope(k, torch.tensor([102.]))).sum()  # posiciones 105 y 102 (distancia 3)
    s3 = (rope(q, torch.tensor([5.])) * rope(k, torch.tensor([0.]))).sum()      # distancia 5
    print(f"RoPE: <q5,k2>={s1:.4f}  <q105,k102>={s2:.4f} (iguales: solo importa la distancia)   <q5,k0>={s3:.4f} (distinta)")

## 3. Un mini-GPT completo (D2L 11.7; Tunstall Cap. 10; Alammar Cap. 3)

Bloque Transformer **pre-LN** (el usado en GPT-2+ y Llama):
$$\mathbf{h} \leftarrow \mathbf{h} + \mathrm{MHA}(\mathrm{LN}(\mathbf{h})),\qquad \mathbf{h} \leftarrow \mathbf{h} + \mathrm{FFN}(\mathrm{LN}(\mathbf{h})),$$
con $\mathrm{FFN}(x)=W_2\,\phi(W_1x)$ (dimensión interna 4× ; Llama usa SwiGLU). La cabeza final proyecta al vocabulario y se entrena con **entropía cruzada para predecir el siguiente token**. Eso es *todo* el pre-entrenamiento de un LLM; la escala (datos, parámetros, cómputo) hace el resto (leyes de escalado de Kaplan/Chinchilla).

Entrenamos a nivel de carácter sobre un texto pequeño para ver el mecanismo (en CPU tarda ~1-2 min).

In [ ]:
if TORCH:
    class Bloque(nn.Module):
        def __init__(self, d, h, dropout=0.1):
            super().__init__(); self.h = h
            self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
            self.qkv = nn.Linear(d, 3 * d, bias=False); self.o = nn.Linear(d, d, bias=False)
            self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)); self.drop = nn.Dropout(dropout)
        def forward(self, x):
            B, T, d = x.shape
            q, k, v = self.qkv(self.ln1(x)).split(d, dim=-1)
            q, k, v = [t.view(B, T, self.h, d // self.h).transpose(1, 2) for t in (q, k, v)]   # (B,h,T,dh)
            pos = torch.arange(T, device=x.device, dtype=torch.float32); q, k = rope(q, pos), rope(k, pos)
            a = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.1 if self.training else 0.0)
            x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, T, d)))
            return x + self.drop(self.ffn(self.ln2(x)))

    class MiniGPT(nn.Module):
        def __init__(self, vocab, d=128, h=4, n_capas=4, ctx=128):
            super().__init__(); self.ctx = ctx
            self.emb = nn.Embedding(vocab, d); self.bloques = nn.Sequential(*[Bloque(d, h) for _ in range(n_capas)])
            self.ln = nn.LayerNorm(d); self.head = nn.Linear(d, vocab, bias=False); self.head.weight = self.emb.weight  # weight tying
        def forward(self, idx):
            return self.head(self.ln(self.bloques(self.emb(idx))))
        @torch.no_grad()
        def generar(self, idx, n, temperatura=1.0, top_p=0.9):
            for _ in range(n):
                logits = self(idx[:, -self.ctx:])[:, -1] / temperatura
                probs = logits.softmax(-1); sp, si = probs.sort(descending=True); cum = sp.cumsum(-1)
                sp[cum - sp > top_p] = 0; sp = sp / sp.sum(-1, keepdim=True)                # muestreo nucleus (top-p)
                idx = torch.cat([idx, si.gather(-1, torch.multinomial(sp, 1))], 1)
            return idx

    texto = ("En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero, "
             "adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero, salpicón las más noches, duelos y quebrantos los sábados, "
             "lantejas los viernes, algún palomino de añadidura los domingos, consumían las tres partes de su hacienda. El resto della concluían sayo de velarte, "
             "calzas de velludo para las fiestas, con sus pantuflos de lo mesmo, y los días de entresemana se honraba con su vellorí de lo más fino. "
             "Tenía en su casa una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte, y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera. "
             "Frisaba la edad de nuestro hidalgo con los cincuenta años; era de complexión recia, seco de carnes, enjuto de rostro, gran madrugador y amigo de la caza. "
             "Quieren decir que tenía el sobrenombre de Quijada, o Quesada, que en esto hay alguna diferencia en los autores que deste caso escriben; "
             "aunque por conjeturas verosímiles se deja entender que se llamaba Quejana. Pero esto importa poco a nuestro cuento; basta que en la narración dél no se salga un punto de la verdad. ") * 6
    chars = sorted(set(texto)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
    data = torch.tensor([stoi[c] for c in texto]); ctx = 64
    print(f"vocabulario: {len(chars)} caracteres, {len(data)} tokens")

    modelo = MiniGPT(len(chars), ctx=ctx).to(device); opt = torch.optim.AdamW(modelo.parameters(), 3e-3, weight_decay=0.1)
    print("parámetros:", sum(p.numel() for p in modelo.parameters()))
    t0 = time.time()
    for step in range(700):
        ix = torch.randint(0, len(data) - ctx - 1, (32,))
        xb = torch.stack([data[i:i+ctx] for i in ix]).to(device); yb = torch.stack([data[i+1:i+ctx+1] for i in ix]).to(device)
        loss = F.cross_entropy(modelo(xb).view(-1, len(chars)), yb.view(-1)); opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0); opt.step()
        if step % 100 == 0: print(f"paso {step:4d}  loss={loss.item():.3f}  ({time.time()-t0:.0f}s)")
    modelo.eval(); inicio = torch.tensor([[stoi[c] for c in "En un lugar"]], device=device)
    print("\n>>>", "".join(itos[i] for i in modelo.generar(inicio, 200, temperatura=0.8)[0].tolist()))

### Lo que hace utilizable a un LLM
* **Tokenización BPE** (Alammar Cap. 2): en vez de caracteres o palabras, sub-palabras aprendidas por frecuencia (~32k-256k tokens). Implementamos el algoritmo de fusión.
* **KV-cache**: al generar, las claves/valores de los tokens pasados no cambian → se guardan y cada nuevo token cuesta $O(T)$ en vez de $O(T^2)$. Es lo que consume la memoria de GPU en inferencia (por eso existen GQA/MQA, *paged attention* en vLLM, cuantización del cache).
* **Decodificación**: *greedy*, *beam*, temperatura, top-k, **top-p**; *speculative decoding* para acelerar.

In [ ]:
from collections import Counter

def entrenar_bpe(corpus, n_merges=30):
    palabras = Counter(tuple(w) + ("</w>",) for w in corpus.split())
    merges = []
    for _ in range(n_merges):
        pares = Counter()
        for w, f in palabras.items():
            for a, b in zip(w, w[1:]): pares[(a, b)] += f
        if not pares: break
        (a, b), _ = pares.most_common(1)[0]; merges.append((a, b))
        nuevo = {}
        for w, f in palabras.items():
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and (w[i], w[i+1]) == (a, b): out.append(a + b); i += 2
                else: out.append(w[i]); i += 1
            nuevo[tuple(out)] = f
        palabras = nuevo
    return merges, palabras

corpus = "el aprendizaje profundo aprende representaciones profundas el aprendiz aprende aprendiendo representar lo aprendido"
merges, vocab = entrenar_bpe(corpus, 12)
print("fusiones aprendidas (en orden):", [a + "+" + b for a, b in merges])
print("segmentación resultante:", [" ".join(w) for w in list(vocab)[:6]])

## 4. Familias de modelos y ciclo de vida de un LLM (Alammar Cap. 1; Tunstall Cap. 1; Fregly Cap. 3)

| Familia | Atención | Objetivo de pre-entrenamiento | Uso típico | Ejemplos |
|---|---|---|---|---|
| **Encoder** | bidireccional | *masked LM* | clasificación, NER, *embeddings* para búsqueda | BERT, RoBERTa, DeBERTa, ModernBERT (2024) |
| **Decoder** | causal | siguiente token | generación, chat, agentes | GPT-4/5, Claude, Llama 3-4, Mistral, Qwen 3, DeepSeek V3/R1, Gemma 3 |
| **Encoder-decoder** | ambas | *span corruption* | traducción, resumen | T5, BART, Whisper (audio) |

**Ciclo de vida** de un LLM de chat (Ouyang et al., 2022; Rafailov et al., 2023):
1. **Pre-entrenamiento** sobre billones de tokens (predicción del siguiente token) → modelo base.
2. **SFT** (*supervised fine-tuning*) con pares instrucción-respuesta → sigue instrucciones.
3. **Alineación por preferencias**: RLHF con PPO (Módulo 8) o **DPO** (sin RL explícito) → útil, honesto, inocuo.
4. (2025-2026) **Razonamiento con RL verificable** (RLVR/GRPO: DeepSeek-R1, o1/o3-style): recompensas automáticas en matemáticas/código → cadenas de pensamiento largas.

**Leyes de escalado**: la pérdida cae como potencia del cómputo; Chinchilla (2022) fijó ~20 tokens/parámetro como óptimo de entrenamiento; hoy se "sobre-entrena" (Llama 3: >200 tokens/parámetro) porque el costo de inferencia domina.

## 5. Modelos de espacio de estados (SSM) y Mamba: la alternativa lineal

Un SSM lineal continuo $\dot h = Ah + Bx,\ y = Ch$ discretizado da una **recurrencia lineal**
$$h_t = \bar A h_{t-1} + \bar B x_t,\qquad y_t = C h_t,$$
que (a) se puede calcular como **convolución** en paralelo durante el entrenamiento y (b) como **recurrencia $O(1)$ por token** en inferencia, sin KV-cache. S4 (Gu et al., 2022) lo hizo funcionar; **Mamba** (Gu & Dao, 2023) hizo $\bar A,\bar B, C$ **dependientes de la entrada** ("selectivos"), recuperando la capacidad de decidir qué recordar y qué olvidar, con un *scan* paralelo eficiente en GPU. **Mamba-2** (2024) unificó SSM y atención lineal (*State Space Duality*); **Mamba-3** (2026) mejora la recurrencia (discretización trapezoidal, estados complejos) y reduce a la mitad el costo de decodificación.

**Dónde estamos en 2026:** ni reemplazo ni fracaso — **híbridos** que intercalan capas de atención (recuerdo exacto) con capas SSM (eficiencia): Jamba, Zamba2, Hymba, Nemotron-H, Falcon-H1, y variantes de atención lineal (RWKV-7, Gated DeltaNet, Qwen3-Next). Regla práctica: la atención gana en *recall* exacto (copiar, buscar una aguja); los SSM ganan en secuencias largas y latencia.

Implementamos un SSM selectivo mínimo (con *scan* secuencial, suficiente para entender) y medimos el escalado del costo frente a la atención.

In [ ]:
if TORCH:
    class SSMSelectivo(nn.Module):
        '''Versión didáctica de una capa Mamba (sin las convoluciones ni el scan paralelo optimizado).'''
        def __init__(self, d, n_estado=16):
            super().__init__(); self.d, self.n = d, n_estado
            self.A_log = nn.Parameter(torch.log(torch.arange(1, n_estado + 1).float()).repeat(d, 1))   # A diagonal negativa (estable)
            self.B = nn.Linear(d, n_estado, bias=False); self.C = nn.Linear(d, n_estado, bias=False)   # dependientes de la entrada
            self.dt = nn.Linear(d, d); self.D = nn.Parameter(torch.ones(d))
        def forward(self, x):                                   # x: (B,T,d)
            Bsz, T, d = x.shape
            A = -self.A_log.exp()                               # (d,n)
            delta = F.softplus(self.dt(x))                      # paso de tiempo por canal y token: (B,T,d) -> controla 'olvido'
            Bx, Cx = self.B(x), self.C(x)                       # (B,T,n)
            h = torch.zeros(Bsz, d, self.n, device=x.device); ys = []
            for t in range(T):                                  # recurrencia O(T): coste constante por token en inferencia
                Abar = torch.exp(delta[:, t, :, None] * A)      # discretización ZOH: (B,d,n)
                Bbar = delta[:, t, :, None] * Bx[:, t, None, :]
                h = Abar * h + Bbar * x[:, t, :, None]
                ys.append((h * Cx[:, t, None, :]).sum(-1) + self.D * x[:, t])
            return torch.stack(ys, 1)

    ssm = SSMSelectivo(64).to(device); mha = nn.MultiheadAttention(64, 4, batch_first=True).to(device)
    Ts, t_ssm, t_att = [64, 128, 256, 512, 1024], [], []
    with torch.no_grad():
        for T in Ts:
            x = torch.randn(4, T, 64, device=device)
            t0 = time.time(); ssm(x); t_ssm.append(time.time() - t0)
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), 1)
            t0 = time.time(); mha(x, x, x, attn_mask=mask); t_att.append(time.time() - t0)
    plt.plot(Ts, t_att, "o-", label="atención (O(T²))"); plt.plot(Ts, t_ssm, "s-", label="SSM selectivo, scan secuencial (O(T))")
    plt.xlabel("longitud T"); plt.ylabel("segundos"); plt.legend(); plt.title("Escalado con la longitud (CPU; el scan optimizado de Mamba es mucho más rápido)"); plt.show()
    print("Nota: en CPU y sin kernels fusionados la constante del bucle Python domina; lo importante es la *forma* de las curvas al crecer T.")

In [ ]:
if TORCH:
    # ¿Aprende un SSM la tarea de memoria (contar 1s) del Módulo 3? Comparación rápida SSM vs Transformer a igual presupuesto.
    def datos_seq(n, T=40):
        x = (torch.rand(n, T, 1) < 0.3).float(); return x, x.sum(dim=(1, 2)).long().clamp(max=20)
    xs, ys = datos_seq(2000); xv, yv = datos_seq(500)

    class Clf(nn.Module):
        def __init__(self, core, d=32):
            super().__init__(); self.inp = nn.Linear(1, d); self.core = core; self.out = nn.Linear(d, 21)
        def forward(self, x):
            h = self.core(self.inp(x)); return self.out(h[:, -1])
    class TransCore(nn.Module):
        def __init__(self, d=32):
            super().__init__(); self.b = Bloque(d, 4, dropout=0.0)
        def forward(self, x): return self.b(x)

    for nombre, core in [("SSM selectivo", SSMSelectivo(32)), ("Transformer (1 bloque)", TransCore())]:
        m = Clf(core).to(device); opt = torch.optim.AdamW(m.parameters(), 3e-3)
        for step in range(400):
            ix = torch.randint(0, 2000, (64,)); loss = F.cross_entropy(m(xs[ix].to(device)), ys[ix].to(device)); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): acc = (m(xv.to(device)).argmax(1).cpu() == yv).float().mean().item()
        print(f"{nombre:24s} exactitud validación = {acc:.3f}")

## 6. Hugging Face en la práctica (Tunstall Cap. 2, 8; Alammar Cap. 1-3)

```python
# OPCIONAL: requiere `uv pip install transformers accelerate` y descarga de pesos (~500 MB)
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
clf = pipeline("text-classification", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")   # encoder multilingüe
print(clf("Este notebook explica la atención con una claridad sorprendente"))

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype="auto", device_map="auto")
msgs = [{"role": "user", "content": "Explica RoPE en dos frases."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
print(tok.decode(llm.generate(ids, max_new_tokens=80, do_sample=True, top_p=0.9, temperature=0.7)[0][ids.shape[1]:]))
print(tok.tokenize("tokenización subpalabra"))     # inspecciona el BPE real
```
* **Inferencia en producción** (Tunstall Cap. 8 + práctica 2026): **vLLM** / SGLang / TensorRT-LLM (paged attention, *continuous batching*), cuantización **GPTQ/AWQ/FP8**, *speculative decoding*, *prefix caching*.
* **Fine-tuning** ligero con LoRA/QLoRA (Módulo 6) usando `peft` + `trl`.

In [ ]:
# OPCIONAL: tokenizador real de Hugging Face (solo descarga el tokenizador, ~2 MB) — protegido si no hay red/librería
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("gpt2")
    for s in ["aprendizaje", "learning", "Transformers are parallelizable", "🦙 Llama"]:
        print(f"{s!r:40s} -> {tok.tokenize(s)}")
except Exception as e:
    print("transformers/red no disponibles en este entorno:", type(e).__name__)

## 📦 Recursos de los repositorios oficiales de los libros

| Libro | Recurso | Para qué usarlo aquí |
|---|---|---|
| Alammar & Grootendorst, *Hands-On LLMs* | [Cap. 1 Introduction to Language Models](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb), [Cap. 2 Tokens and Token Embeddings](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb), [Cap. 3 Looking Inside LLMs](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter03/Chapter%203%20-%20Looking%20Inside%20LLMs.ipynb) · bonus: [*A Visual Guide to Mamba*](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-mamba-and-state), [*A Visual Guide to Quantization*](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-quantization), [*A Visual Guide to Mixture of Experts*](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-mixture-of-experts), [*A Visual Guide to Reasoning LLMs*](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-reasoning-llms), [*The Illustrated DeepSeek-R1*](https://newsletter.languagemodels.co/p/the-illustrated-deepseek-r1) | Los notebooks corren en Colab (T4 gratuita). La guía visual de **Mamba** complementa nuestra sección 5; la de **MoE** cubre la arquitectura de Mixtral/DeepSeek que no implementamos |
| Tunstall et al., *NLP with Transformers* | [`01_introduction`](https://colab.research.google.com/github/nlp-with-transformers/notebooks/blob/main/01_introduction.ipynb), [`02_classification`](https://colab.research.google.com/github/nlp-with-transformers/notebooks/blob/main/02_classification.ipynb), [`03_transformer-anatomy`](https://colab.research.google.com/github/nlp-with-transformers/notebooks/blob/main/03_transformer-anatomy.ipynb) (atención desde cero, como nuestra sección 1), [`08_model-compression`](https://colab.research.google.com/github/nlp-with-transformers/notebooks/blob/main/08_model-compression.ipynb) (destilación, cuantización, ONNX), [`10_transformers-from-scratch`](https://colab.research.google.com/github/nlp-with-transformers/notebooks/blob/main/10_transformers-from-scratch.ipynb) (entrena un GPT de código con `datasets` + `accelerate`) | El Cap. 10 escala nuestro `MiniGPT` a un entrenamiento real; el Cap. 8 es la referencia práctica de eficiencia en inferencia |
| Zhang et al., *D2L* | [Cap. 11 Attention & Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html): *queries-keys-values*, *attention-scoring-functions*, *multihead-attention*, *self-attention-and-positional-encoding*, *transformer*, *vision-transformer*, *large-pretraining-transformers* · [Cap. 15-16 NLP](https://d2l.ai/chapter_natural-language-processing-pretraining/index.html) (word2vec, BERT desde cero) | Derivaciones paso a paso con código PyTorch; la sección *large-pretraining-transformers* explica encoder/decoder/enc-dec y leyes de escalado |
| Géron, *Hands-On ML 3* | [`16_nlp_with_rnns_and_attention.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/16_nlp_with_rnns_and_attention.ipynb) | Traducción con atención y Transformer en Keras; Vision Transformer |
| Fregly et al. | [`03_foundation/`](https://github.com/generative-ai-on-aws/generative-ai-on-aws/tree/main/03_foundation) | Notebooks de exploración de modelos fundacionales en SageMaker JumpStart |

## 7. Ejercicios
1. **Cabezas de atención.** Extrae los pesos de atención de `MiniGPT` (modifica `Bloque` para devolverlos) y visualiza, para una frase, qué caracteres atiende cada cabeza de la última capa.
2. **Ablación posicional.** Entrena `MiniGPT` (a) sin RoPE ni posición, (b) con sinusoidal sumada, (c) con RoPE. Compara la pérdida final. ¿Por qué (a) aún aprende algo? (pista: la máscara causal filtra información posicional).
3. **KV-cache.** Implementa `generar` con cache de $K,V$ por capa y mide el tiempo para generar 500 tokens frente a la versión sin cache.
4. **Escalado.** Entrena `MiniGPT` con `d` ∈ {32, 64, 128, 256} durante el mismo número de pasos y grafica la pérdida final vs parámetros en escala log-log. ¿Ves una ley de potencias?
5. **Memoria larga.** Repite la tarea de conteo con `T=200` para RNN (Módulo 3), Transformer y SSM. Luego cambia la tarea a **"copiar el token de la posición k"** (recall exacto). ¿Qué arquitectura gana en cada caso? Relaciónalo con los híbridos.
6. **Embeddings.** Con `sentence-transformers` (o el `encoder` de SimCLR del Módulo 4 como analogía) calcula similitud coseno entre frases parafraseadas y no relacionadas. Esto es la base del RAG del Módulo 6.

## Referencias
* Zhang, A. et al. *Dive into Deep Learning*. Cap. 11, 15-16.
* Tunstall, L., von Werra, L. & Wolf, T. (2022). *Natural Language Processing with Transformers* (ed. rev.). O'Reilly. Cap. 1-3, 8, 10. Código: github.com/nlp-with-transformers/notebooks
* Alammar, J. & Grootendorst, M. (2024). *Hands-On Large Language Models*. O'Reilly. Cap. 1-3. Código: github.com/HandsOnLLM/Hands-On-Large-Language-Models
* Géron, A. (2022). *Hands-On ML* (3.ª ed.). Cap. 16.
* Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
* Su, J. et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding*.
* Dao, T. et al. (2022). *FlashAttention*. NeurIPS. · Kwon, W. et al. (2023). *Efficient Memory Management for LLM Serving with PagedAttention* (vLLM). SOSP.
* Hoffmann, J. et al. (2022). *Training Compute-Optimal Large Language Models* (Chinchilla).
* Ouyang, L. et al. (2022). *Training language models to follow instructions with human feedback*. · Rafailov, R. et al. (2023). *Direct Preference Optimization*. · DeepSeek-AI (2025). *DeepSeek-R1*.
* Gu, A. & Dao, T. (2023). *Mamba: Linear-Time Sequence Modeling with Selective State Spaces*. · Dao, T. & Gu, A. (2024). *Transformers are SSMs* (Mamba-2). ICML. · Mamba-3 (2026), Princeton PLI, arXiv:2603.15569.
* Karpathy, A. *nanoGPT* / *Let's build GPT* (video): implementación de referencia minimalista.